# Variance partitioning — family-level results, all patients
gpt2-xl L36, worddur window, pc=100, xcirc null. Hippocampus + ACC, self/other.
Compares semantic encoding against: all controls combined, lexical only, syntactic only, acoustic only.

In [ ]:
import os, glob, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})

VP_DIR  = '/scratch/aniluchavez/ConvoDATAS/VPResults/gpt2-xl_ctx200_worddur_xcirc/pc100'
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

REGIONS    = ['hippocampus', 'ACC']
CONDITIONS = ['self', 'other']
COMPARISONS = ['all_controls', 'lexical', 'syntactic', 'acoustic']
COMP_LABELS = {'all_controls': 'vs all controls', 'lexical': 'vs lexical',
               'syntactic': 'vs syntactic', 'acoustic': 'vs acoustic'}
COND_COLORS = {'self': '#2166ac', 'other': '#d6604d'}

rows = []
for f in sorted(glob.glob(os.path.join(VP_DIR, 'PTY*_L36_VP.pkl'))):
    rows.append(pickle.load(open(f, 'rb')))
data = pd.concat(rows, ignore_index=True)
print(f'{len(data)} rows, {data["patient"].nunique()} patients')
data.head()

In [ ]:
# ── Summary table: region x condition x comparison ───────────────────────────
summary_rows = []
for region in REGIONS:
    for cond in CONDITIONS:
        sub = data[(data['region'] == region) & (data['condition'] == cond)]
        if sub.empty: continue
        n = len(sub)
        summary_rows.append({
            'region': region, 'condition': cond, 'comparison': 'all_controls',
            'n_neurons': n, 'pct_sig': 100 * sub['significant'].mean(),
            'median_unique_sem': sub['unique_semantic'].median(),
            'median_r2_confound': sub['r2_controls'].median(),
        })
        for fam in ['lexical', 'syntactic', 'acoustic']:
            summary_rows.append({
                'region': region, 'condition': cond, 'comparison': fam,
                'n_neurons': n, 'pct_sig': 100 * sub[f'significant_{fam}'].mean(),
                'median_unique_sem': sub[f'unique_semantic_{fam}'].median(),
                'median_r2_confound': sub[f'r2_{fam}'].median(),
            })
summary = pd.DataFrame(summary_rows)
summary

In [ ]:
# ── PLOT 1: % significant by comparison type, faceted by region ─────────────
fig, axes = plt.subplots(1, len(REGIONS), figsize=(6.5 * len(REGIONS), 5), squeeze=False)
x = np.arange(len(COMPARISONS))
w = 0.35

for ax, region in zip(axes[0], REGIONS):
    for i, cond in enumerate(CONDITIONS):
        sub = summary[(summary['region'] == region) & (summary['condition'] == cond)]
        sub = sub.set_index('comparison').loc[COMPARISONS]
        ax.bar(x + (i - 0.5) * w, sub['pct_sig'], width=w, label=cond, color=COND_COLORS[cond])
    ax.axhline(5, color='gray', linewidth=0.8, linestyle='--', label='5% chance')
    ax.set_xticks(x)
    ax.set_xticklabels([COMP_LABELS[c] for c in COMPARISONS], rotation=20, ha='right')
    ax.set_ylabel('% significant neurons')
    ax.set_title(region)
    ax.set_ylim(0, 95)
    ax.legend(frameon=False, fontsize=9)

plt.suptitle('Semantic encoding significance, by confound family controlled for', y=1.04)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/07_pct_sig_by_family.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 2: per-patient % significant (vs all controls) — consistency check ─
fig, axes = plt.subplots(1, len(REGIONS), figsize=(7 * len(REGIONS), 5), squeeze=False)

for ax, region in zip(axes[0], REGIONS):
    per_patient = []
    for cond in CONDITIONS:
        sub = data[(data['region'] == region) & (data['condition'] == cond)]
        for pat, grp in sub.groupby('patient'):
            per_patient.append({'patient': pat, 'condition': cond,
                                 'pct_sig': 100 * grp['significant'].mean(), 'n': len(grp)})
    pp = pd.DataFrame(per_patient)
    if pp.empty: continue
    patients = sorted(pp['patient'].unique())
    xpos = np.arange(len(patients))
    for i, cond in enumerate(CONDITIONS):
        vals = [pp[(pp['patient'] == p) & (pp['condition'] == cond)]['pct_sig'].values for p in patients]
        vals = [v[0] if len(v) else np.nan for v in vals]
        ax.scatter(xpos + (i - 0.5) * 0.15, vals, color=COND_COLORS[cond], label=cond, s=50, zorder=3)
    ax.axhline(5, color='gray', linewidth=0.8, linestyle='--')
    ax.set_xticks(xpos)
    ax.set_xticklabels(patients, rotation=60, ha='right', fontsize=8)
    ax.set_ylabel('% significant (vs all controls)')
    ax.set_title(region)
    ax.set_ylim(-5, 105)
    ax.legend(frameon=False)

plt.suptitle('Per-patient significance — vs all controls', y=1.04)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/07_pct_sig_per_patient.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 3: distribution of unique_semantic (vs all controls), by region x condition
fig, ax = plt.subplots(figsize=(7, 5))
plot_data, labels, colors = [], [], []
for region in REGIONS:
    for cond in CONDITIONS:
        sub = data[(data['region'] == region) & (data['condition'] == cond) & data['significant']]
        plot_data.append(sub['unique_semantic'].dropna().values)
        labels.append(f'{region}\n{cond}')
        colors.append(COND_COLORS[cond])

parts = ax.violinplot(plot_data, showmedians=True)
for pc, c in zip(parts['bodies'], colors):
    pc.set_facecolor(c); pc.set_alpha(0.6)
ax.set_xticks(np.arange(1, len(labels) + 1))
ax.set_xticklabels(labels)
ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
ax.set_ylabel('unique semantic R² (significant neurons only)')
ax.set_title('Effect size distribution — significant neurons, vs all controls')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/07_unique_sem_distribution.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 4: median R² of each family alone vs semantic — explanatory power ──
fig, axes = plt.subplots(1, len(REGIONS), figsize=(6.5 * len(REGIONS), 5), squeeze=False)
fams = ['lexical', 'syntactic', 'acoustic']

for ax, region in zip(axes[0], REGIONS):
    x = np.arange(len(fams) + 1)
    w = 0.35
    for i, cond in enumerate(CONDITIONS):
        sub = data[(data['region'] == region) & (data['condition'] == cond)]
        vals = [sub['r2_semantic'].median()] + [sub[f'r2_{f}'].median() for f in fams]
        ax.bar(x + (i - 0.5) * w, vals, width=w, color=COND_COLORS[cond], label=cond)
    ax.axhline(0, color='k', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(['semantic'] + fams, rotation=20, ha='right')
    ax.set_ylabel('median R² (model alone)')
    ax.set_title(region)
    ax.legend(frameon=False)

plt.suptitle('How much does each predictor family explain on its own?', y=1.04)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/07_r2_alone_by_family.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
display_summary = summary.copy()
display_summary['pct_sig'] = display_summary['pct_sig'].round(1)
display_summary['median_unique_sem'] = display_summary['median_unique_sem'].round(4)
display_summary['median_r2_confound'] = display_summary['median_r2_confound'].round(4)
display_summary

## Variance partitioning — stacked bars (semantic vs. all controls)\nMean R² (clipped ≥ 0) per region x condition, same style as 03_linguistic_confounds.ipynb.

In [ ]:
import matplotlib.patches as mpatches

COMBOS = [('hippocampus', 'self'), ('hippocampus', 'other'), ('ACC', 'self'), ('ACC', 'other')]
PAL = {'unique_semantic': '#2166ac', 'unique_controls': '#d6604d', 'shared': '#92c5de'}
VP_COLS = ['unique_semantic', 'unique_controls', 'shared']

fig, axes = plt.subplots(1, 4, figsize=(13, 5), sharey=True)
for ax, (region, cond) in zip(axes, COMBOS):
    sub = data[(data['region'] == region) & (data['condition'] == cond)]
    if sub.empty:
        ax.set_title(f'{region}/{cond}\n(no data)'); continue

    means = {c: sub[c].clip(lower=0).mean() for c in VP_COLS if c in sub.columns}
    pct_sig = 100 * sub['significant'].mean()

    bottom = 0
    for col in VP_COLS:
        v = means.get(col, 0)
        ax.bar(0, v, bottom=bottom, color=PAL[col], width=0.6)
        if v > 0.0005:
            ax.text(0, bottom + v/2, f'{v:.4f}',
                    ha='center', va='center', fontsize=8.5, color='white', fontweight='bold')
        bottom += v

    ax.set_title(f'{region}\n{cond}\n{pct_sig:.0f}% sig unique sem', fontsize=10)
    ax.set_xticks([])
    ax.set_xlim(-0.5, 0.5)

axes[0].set_ylabel('Mean R² (clipped ≥ 0)')
handles = [mpatches.Patch(facecolor=PAL[c], label=c.replace('_', ' ')) for c in VP_COLS]
fig.legend(handles=handles, loc='lower center', ncol=3,
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.06))
plt.suptitle('gpt2-xl L36  |  5-fold temporal block CV  |  Variance partitioning', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/07_vp_bars.pdf', bbox_inches='tight')
plt.show()

## Partitioned bar chart — significant-neuron counts across confound families\nRaw counts of neurons that pass the unique-semantic significance test against each family alone (lexical / syntactic / acoustic), broken into the 8 mutually-exclusive overlap categories.

In [ ]:
CATS = ['none', 'lex only', 'syn only', 'aco only',
        'lex+syn', 'lex+aco', 'syn+aco', 'all three']
CAT_COLOR = '#4393c3'

fig, axes = plt.subplots(2, 2, figsize=(7 * 2, 5 * 2))

panel_specs = [(r, c) for r in REGIONS for c in CONDITIONS]
for ax, (region, cond) in zip(axes.flat, panel_specs):
    sub = data[(data['region'] == region) & (data['condition'] == cond)]
    lex = sub['significant_lexical'].values
    syn = sub['significant_syntactic'].values
    aco = sub['significant_acoustic'].values
    n = len(sub)

    counts = {
        'none':      (~lex & ~syn & ~aco).sum(),
        'lex only':  ( lex & ~syn & ~aco).sum(),
        'syn only':  (~lex &  syn & ~aco).sum(),
        'aco only':  (~lex & ~syn &  aco).sum(),
        'lex+syn':   ( lex &  syn & ~aco).sum(),
        'lex+aco':   ( lex & ~syn &  aco).sum(),
        'syn+aco':   (~lex &  syn &  aco).sum(),
        'all three': ( lex &  syn &  aco).sum(),
    }
    assert sum(counts.values()) == n

    pct = [100 * counts[c] / n for c in CATS]
    bars = ax.bar(CATS, pct, color=CAT_COLOR)
    for b, c in zip(bars, CATS):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1,
                f'{counts[c]}', ha='center', fontsize=9)
    ax.set_xticklabels(CATS, rotation=35, ha='right')
    ax.set_ylabel('% of neurons')
    ax.set_title(f'{region} / {cond}  (n={n})')
    ax.set_ylim(0, max(pct) * 1.2)

plt.suptitle('Which families a neuron survives controlling for (unique-semantic significance)', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/07_partitioned_bars_significant_neurons.pdf', bbox_inches='tight')
plt.show()

## Variance partitioning — stacked bars, significant neurons only\nSame style as above, but the mean R² is taken only over neurons that pass the unique-semantic significance test (vs. all controls) — shows effect size among the neurons that "count," rather than diluting with non-significant ones.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 5), sharey=True)
for ax, (region, cond) in zip(axes, COMBOS):
    sub_all = data[(data['region'] == region) & (data['condition'] == cond)]
    sub = sub_all[sub_all['significant']]
    if sub.empty:
        ax.set_title(f'{region}/{cond}\n(no sig neurons)'); continue

    means = {c: sub[c].clip(lower=0).mean() for c in VP_COLS if c in sub.columns}
    pct_sig = 100 * len(sub) / len(sub_all)

    bottom = 0
    for col in VP_COLS:
        v = means.get(col, 0)
        ax.bar(0, v, bottom=bottom, color=PAL[col], width=0.6)
        if v > 0.0005:
            ax.text(0, bottom + v/2, f'{v:.4f}',
                    ha='center', va='center', fontsize=8.5, color='white', fontweight='bold')
        bottom += v

    ax.set_title(f'{region}\n{cond}\nn={len(sub)} ({pct_sig:.0f}% of {len(sub_all)})', fontsize=10)
    ax.set_xticks([])
    ax.set_xlim(-0.5, 0.5)

axes[0].set_ylabel('Mean R² (clipped ≥ 0), significant neurons only')
handles = [mpatches.Patch(facecolor=PAL[c], label=c.replace('_', ' ')) for c in VP_COLS]
fig.legend(handles=handles, loc='lower center', ncol=3,
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.06))
plt.suptitle('gpt2-xl L36  |  5-fold temporal block CV  |  Variance partitioning (significant neurons)', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/07_vp_bars_significant.pdf', bbox_inches='tight')
plt.show()

## Variance partitioning — stacked bars, by control family
Same stacked-bar decomposition (unique semantic / unique control / shared) computed separately against each family (lexical, syntactic, acoustic) and against all controls combined — shows whether the "shared" / "unique" split looks different depending on which confound is being controlled for. Derived algebraically from existing columns: `unique_X = unique_semantic_X + r2_X - r2_semantic`, `shared_X = r2_semantic - unique_semantic_X`.

In [ ]:
def vp_components(sub, comparison):
    r2_sem = sub['r2_semantic']
    if comparison == 'all_controls':
        u_sem, r2_x, sig = sub['unique_semantic'], sub['r2_controls'], sub['significant']
    else:
        u_sem  = sub[f'unique_semantic_{comparison}']
        r2_x   = sub[f'r2_{comparison}']
        sig    = sub[f'significant_{comparison}']
    # exclude neurons with degenerate Poisson fits (pseudo-R2 blows up when the
    # saturated/null deviance gap is ~0, e.g. near-zero spike counts in a fold)
    valid = r2_x.between(-2, 2) & r2_sem.between(-2, 2) & u_sem.between(-2, 2)
    n_excluded = (~valid).sum()
    u_sem, r2_x, r2_sem, sig = u_sem[valid], r2_x[valid], r2_sem[valid], sig[valid]
    u_ctrl = u_sem + r2_x - r2_sem
    shared = r2_sem - u_sem
    return u_sem, u_ctrl, shared, sig, n_excluded

fig, axes = plt.subplots(len(COMPARISONS), 4, figsize=(13, 4.2 * len(COMPARISONS)), sharey='row')
for row, comparison in enumerate(COMPARISONS):
    for ax, (region, cond) in zip(axes[row], COMBOS):
        sub = data[(data['region'] == region) & (data['condition'] == cond)]
        if sub.empty:
            ax.set_title(f'{region}/{cond}\n(no data)'); continue

        u_sem, u_ctrl, shared, sig, n_excl = vp_components(sub, comparison)
        means = {'unique_semantic': u_sem.clip(lower=0).mean(),
                 'unique_controls': u_ctrl.clip(lower=0).mean(),
                 'shared': shared.clip(lower=0).mean()}
        pct_sig = 100 * sig.mean()

        bottom = 0
        for col in VP_COLS:
            v = means[col]
            ax.bar(0, v, bottom=bottom, color=PAL[col], width=0.6)
            if v > 0.0005:
                ax.text(0, bottom + v/2, f'{v:.4f}',
                        ha='center', va='center', fontsize=8, color='white', fontweight='bold')
            bottom += v

        excl_note = f', {n_excl} excl.' if n_excl else ''
        title = f'{region}\n{cond}\n{pct_sig:.0f}% sig{excl_note}' if row == 0 else f'{pct_sig:.0f}% sig{excl_note}'
        ax.set_title(title, fontsize=9.5)
        ax.set_xticks([])
        ax.set_xlim(-0.5, 0.5)
    axes[row, 0].set_ylabel(f'{COMP_LABELS[comparison]}\nMean R² (clipped ≥ 0)', fontsize=9.5)

handles = [mpatches.Patch(facecolor=PAL[c], label=c.replace('_', ' ')) for c in VP_COLS]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.01))
plt.suptitle('gpt2-xl L36  |  Variance partitioning, by control family', y=1.0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/07_vp_bars_by_family.pdf', bbox_inches='tight')
plt.show()